<a href="https://colab.research.google.com/github/Dk-221/secure-document-verfication-system/blob/main/secure_documentation_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask flask-cors pyngrok sqlalchemy python-jose passlib[bcrypt] pillow pytesseract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 33.6 MB/s eta 0:00:00


In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Float, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime

engine = create_engine('sqlite:///securedocs.db')
Base = declarative_base()

class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    name = Column(String(100))
    email = Column(String(100), unique=True)
    password_hash = Column(String(200))
    role = Column(String(20), default='user')
    created_at = Column(DateTime, default=datetime.utcnow)

class Document(Base):
    __tablename__ = 'documents'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer)
    filename = Column(String(200))
    file_hash = Column(String(64))
    status = Column(String(20), default='pending')
    ocr_text = Column(Text)
    forgery_score = Column(Float, default=0.0)
    created_at = Column(DateTime, default=datetime.utcnow)

class Verification(Base):
    __tablename__ = 'verifications'
    id = Column(Integer, primary_key=True)
    document_id = Column(Integer)
    blockchain_tx = Column(String(200))
    result = Column(String(20))
    verified_at = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
print("Database and tables created successfully!")

Database and tables created successfully!


/tmp/ipykernel_422/2093564873.py:7: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [ ]:
from passlib.context import CryptContext
from jose import jwt
from datetime import datetime, timedelta

SECRET_KEY = "your-secret-key-change-this"
ALGORITHM = "HS256"
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password):
    return pwd_context.hash(password)

def verify_password(plain, hashed):
    return pwd_context.verify(plain, hashed)

def create_token(data):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(hours=24)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token):
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

print("Auth system ready!")

Auth system ready!


In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import hashlib
import os

app = Flask(__name__)
CORS(app)

UPLOAD_FOLDER = 'uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "message": "API is running"})

@app.route('/auth/register', methods=['POST'])
def register():
    data = request.json
    session = Session()
    existing = session.query(User).filter_by(email=data['email']).first()
    if existing:
        return jsonify({"error": "Email already exists"}), 400
    user = User(
        name=data['name'],
        email=data['email'],
        password_hash=hash_password(data['password'])
    )
    session.add(user)
    session.commit()
    token = create_token({"user_id": user.id, "email": user.email})
    return jsonify({"message": "User created", "token": token})

@app.route('/auth/login', methods=['POST'])
def login():
    data = request.json
    session = Session()
    user = session.query(User).filter_by(email=data['email']).first()
    if not user or not verify_password(data['password'], user.password_hash):
        return jsonify({"error": "Invalid credentials"}), 401
    token = create_token({"user_id": user.id, "email": user.email})
    return jsonify({"token": token, "name": user.name})

@app.route('/documents/upload', methods=['POST'])
def upload_document():
    if 'file' not in request.files:
        return jsonify({"error": "No file provided"}), 400
    file = request.files['file']
    file_content = file.read()
    file_hash = hashlib.sha256(file_content).hexdigest()
    filepath = os.path.join(UPLOAD_FOLDER, file.filename)
    with open(filepath, 'wb') as f:
        f.write(file_content)
    session = Session()
    doc = Document(
        filename=file.filename,
        file_hash=file_hash,
        status='pending'
    )
    session.add(doc)
    session.commit()
    return jsonify({
        "message": "Document uploaded",
        "document_id": doc.id,
        "file_hash": file_hash,
        "status": doc.status
    })

@app.route('/documents', methods=['GET'])
def get_documents():
    session = Session()
    docs = session.query(Document).all()
    return jsonify([{
        "id": d.id,
        "filename": d.filename,
        "file_hash": d.file_hash,
        "status": d.status,
        "forgery_score": d.forgery_score,
        "created_at": str(d.created_at)
    } for d in docs])

@app.route('/verify/<file_hash>', methods=['GET'])
def verify_document(file_hash):
    session = Session()
    doc = session.query(Document).filter_by(file_hash=file_hash).first()
    if not doc:
        return jsonify({"verified": False, "message": "Document not found"})
    return jsonify({
        "verified": True,
        "filename": doc.filename,
        "status": doc.status,
        "uploaded_at": str(doc.created_at)
    })

print("Flask API ready!")

Flask API ready!


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BCNxx10flgvlZZWOKA54CjxLHh_7u4nii4pD84AdFKocjPhF")
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

app.run(port=5000)

Public URL: NgrokTunnel: "https://flaggingly-collinear-pamelia.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
ERROR:pyngrok.process.ngrok:t=2026-03-20T07:14:06+0000 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=6cf46b8da0b3 clientid=d4ef480cb80caf2a578558f815c4b573
ERROR:pyngrok.process.ngrok:t=2026-03-20T07:14:06+0000 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=6ed336d3bc35 err="session closed"
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 07:33:46] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 07:33:47] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 07:35:48] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 07:35:50] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 07:36:00] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1

In [ ]:
response = requests.get("https://flaggingly-collinear-pamelia.ngrok-free.dev/health")
print(response.json())

In [ ]:
import requests

base_url = "https://flaggingly-collinear-pamelia.ngrok-free.dev"

response = requests.post(f"{base_url}/auth/register",
    json={
        "name": "Milan Kothari",
        "email": "milan@test.com",
        "password": "test123"
    },
    headers={"Content-Type": "application/json"}
)
print(response.status_code)
print(response.json())